# AI Image Detector - Google Colab Setup Guide

This notebook sets up the AI Image Detector project for training on Google Colab with GPU acceleration.

**Prerequisites:**
- Your project pushed to GitHub (code only)
- `data.zip` uploaded to Google Drive at `My Drive/ai-image-detector-data/data.zip`

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Section 1: Install Required Libraries

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q timm
!pip install -q scikit-learn
!pip install -q pillow
!pip install -q opencv-python
!pip install -q matplotlib
!pip install -q seaborn
!pip install -q numpy
!pip install -q tqdm
!pip install -q python-dotenv
!pip install -q git+https://github.com/facebookresearch/pytorch-grad-cam.git

print("✓ All dependencies installed successfully!")

## Section 2: Clone GitHub Repository

Replace `YOUR_GITHUB_URL` with your actual GitHub repository URL

In [ ]:
import os
from pathlib import Path

# CHANGE THIS TO YOUR GITHUB REPOSITORY URL
GITHUB_URL = "https://github.com/YOUR_USERNAME/ai-image-detector.git"

# Clone the repository
if not os.path.exists('/content/ai-image-detector'):
    !git clone {GITHUB_URL} /content/ai-image-detector
    print("✓ Repository cloned successfully!")
else:
    print("✓ Repository already exists!")

# Change to project directory
os.chdir('/content/ai-image-detector')
print(f"Current directory: {os.getcwd()}")

## Section 3: Mount Drive and Extract Data

This cell mounts your Google Drive and automatically unzips the `data.zip` file into the project folder.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Path to your zip file on Drive
ZIP_PATH = "/content/drive/MyDrive/ai-image-detector-data/data.zip"
PROJECT_PATH = "/content/ai-image-detector"

# 3. Extract data
if os.path.exists(ZIP_PATH):
    print("📦 Extracting data... this may take a few minutes.")
    !unzip -q {ZIP_PATH} -d {PROJECT_PATH}
    print("✓ Data extracted successfully!")
else:
    print(f"✗ Error: Could not find zip file at {ZIP_PATH}")
    print("Please verify the path in your Google Drive.")

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"


## Section 4: Verify Dataset and Configuration

In [ ]:
import sys
from pathlib import Path

# Add project to path
sys.path.insert(0, '/content/ai-image-detector')

# Import config
from config import (
    DATA_DIR, TRAIN_DIR, VALID_DIR, TEST_DIR,
    OUTPUT_DIR, CHECKPOINT_DIR, PLOT_DIR, GRADCAM_DIR,
    DEVICE, IMG_SIZE, BATCH_SIZE, MODEL_NAME
)

print("📊 Configuration Summary:")
print(f"  Device: {DEVICE}")
print(f"  Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Model: {MODEL_NAME}")

print("\n✓ Verifying data exists:")
def count_images(directory):
    if directory.exists():
        return len(list(directory.glob("**/*.jpg"))) + len(list(directory.glob("**/*.png")))
    return 0

print(f"  Train images: {count_images(TRAIN_DIR)}")
print(f"  Valid images: {count_images(VALID_DIR)}")
print(f"  Test images: {count_images(TEST_DIR)}")

if count_images(TRAIN_DIR) == 0:
    print("\n⚠️ WARNING: No images found in training directory!")
    print(f"Check if your zip extracted into: {DATA_DIR}")
    print("Project Structure:")
    !ls -R {DATA_DIR} | head -n 20

# Create output directories
for path in [OUTPUT_DIR, CHECKPOINT_DIR, PLOT_DIR, GRADCAM_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Section 5: Start Training

Run the training script. This will use the GPU if available.

In [ ]:
!python src/train.py

## Section 6: Backup Checkpoints to Drive

Copy trained checkpoints to Google Drive so they aren't lost when the session ends.

In [ ]:
import shutil
from pathlib import Path

DRIVE_SAVE_PATH = Path('/content/drive/MyDrive/ai-image-detector-data/checkpoints')
DRIVE_SAVE_PATH.mkdir(parents=True, exist_ok=True)

if CHECKPOINT_DIR.exists():
    shutil.copytree(CHECKPOINT_DIR, DRIVE_SAVE_PATH, dirs_exist_ok=True)
    print(f"✓ Checkpoints backed up to Drive: {DRIVE_SAVE_PATH}")